In [ ]:
import time
# from jimgw.jim import Jim
# import flowMC.sampler
from jimgw import single_event
# import jimgw.single_event

from jimgw.single_event.detector import H1, L1
from jimgw.single_event.likelihood import (
    HeterodynedTransientLikelihoodFD,
    TransientLikelihoodFD,
)
from jimgw.single_event.waveform import RippleIMRPhenomD
from jimgw.prior import Uniform, Composite
import jax.numpy as jnp
import jax

jax.config.update("jax_enable_x64", True)

###########################################
########## First we grab data #############
###########################################

total_time_start = time.time()

# first, fetch a 4s segment centered on GW150914
gps = 1126259462.4
duration = 4
post_trigger_duration = 2
start_pad = duration - post_trigger_duration
end_pad = post_trigger_duration
fmin = 20.0
fmax = 1024.0

ifos = ["H1", "L1"]

H1.load_data(gps, start_pad, end_pad, fmin, fmax, psd_pad=16, tukey_alpha=0.2)
L1.load_data(gps, start_pad, end_pad, fmin, fmax, psd_pad=16, tukey_alpha=0.2)

Mc_prior = Uniform(10.0, 80.0, naming=["M_c"])
q_prior = Uniform(
    0.125,
    1.0,
    naming=["q"],
    transforms={"q": ("eta", lambda params: params["q"] / (1 + params["q"]) ** 2)},
)
s1z_prior = Uniform(-1.0, 1.0, naming=["s1_z"])
s2z_prior = Uniform(-1.0, 1.0, naming=["s2_z"])
dL_prior = Uniform(0.0, 2000.0, naming=["d_L"])
t_c_prior = Uniform(-0.05, 0.05, naming=["t_c"])
phase_c_prior = Uniform(0.0, 2 * jnp.pi, naming=["phase_c"])
cos_iota_prior = Uniform(
    -1.0,
    1.0,
    naming=["cos_iota"],
    transforms={
        "cos_iota": (
            "iota",
            lambda params: jnp.arccos(
                jnp.arcsin(jnp.sin(params["cos_iota"] / 2 * jnp.pi)) * 2 / jnp.pi
            ),
        )
    },
)
psi_prior = Uniform(0.0, jnp.pi, naming=["psi"])
ra_prior = Uniform(0.0, 2 * jnp.pi, naming=["ra"])
sin_dec_prior = Uniform(
    -1.0,
    1.0,
    naming=["sin_dec"],
    transforms={
        "sin_dec": (
            "dec",
            lambda params: jnp.arcsin(
                jnp.arcsin(jnp.sin(params["sin_dec"] / 2 * jnp.pi)) * 2 / jnp.pi
            ),
        )
    },
)

prior = Composite(
    [
        Mc_prior,
        q_prior,
        s1z_prior,
        s2z_prior,
        dL_prior,
        t_c_prior,
        phase_c_prior,
        cos_iota_prior,
        psi_prior,
        ra_prior,
        sin_dec_prior,
    ]
)

bounds = jnp.array(
    [
        [10.0, 80.0],
        [0.125, 1.0],
        [-1.0, 1.0],
        [-1.0, 1.0],
        [0.0, 2000.0],
        [-0.05, 0.05],
        [0.0, 2 * jnp.pi],
        [-1.0, 1.0],
        [0.0, jnp.pi],
        [0.0, 2 * jnp.pi],
        [-1.0, 1.0],
    ]
)

likelihood = HeterodynedTransientLikelihoodFD(
    [H1, L1],
    prior=prior,
    bounds=bounds,
    waveform=RippleIMRPhenomD(),
    trigger_time=gps,
    duration=duration,
    post_trigger_duration=post_trigger_duration,
    n_loops=300,
);

In [ ]:
def potential(x):
    """ 
    Function wrapper for jjim
    """
    param = {'M_c': x[0],
            'eta': x[1],
            's1_z': x[2],
            's2_z': x[3],
            'd_L': x[4],
            't_c': x[5],
            'phase_c': x[6],
            'iota': x[7],
            'psi': x[8],
            'ra': x[9],
            'dec': x[10]}

    # QUESTION: What parameters do we use in evaluating the likelihood here?
    return -likelihood.evaluate(param, {}) #- prior.log_prob(param)

In [ ]:
param_labels = ['M_c',
                'eta',
                's1_z',
                's2_z',
                'd_L',
                't_c',
                'phase_c',
                'iota',
                'psi',
                'ra',
                'dec']

In [ ]:
import numpy as np

prior_bounds = jnp.array(
    [
        [10.0, 80.0],              # Mc
        [0.240, 0.249],            # eta
        [-1.0, 1.0],               # s1_z
        [-1.0, 1.0],               # s2_z
        [0.0, 2000.0],             # d_L
        [-0.05, 0.05],             # t_c
        [0.0, 2 * jnp.pi],         # phase_c
        [0.0, jnp.pi],             # iota
        [0.0, jnp.pi],             # psi
        [0.0, 2 * jnp.pi],         # ra
        [-jnp.pi/2, jnp.pi/2],     # dec 
    ]
)

DoF = 11
n = 3
prior_samples = np.zeros((n, DoF))
a = np.zeros(DoF)
b = np.zeros(DoF)

for i in range(DoF):
    prior_samples[:, i] = np.random.uniform(low=prior_bounds[i][0], high=prior_bounds[i][1], size=n)
    a[i] = prior_bounds[i][0]
    b[i] = prior_bounds[i][1]

In [ ]:
periodic_coordinates = jnp.array([])
bounded_coordinates = jnp.array([0, 1, 2, 3, 4, 5, 6, ])

In [ ]:
true_params = np.zeros(11)
i = 0
for key in likelihood.ref_params:
    if key != 'gmst':
        true_params[i] = likelihood.ref_params[key] 
        i += 1

In [ ]:
import os, sys

sys.path.append("...")
sys.path.append(".")
sys.path.append("..")

In [ ]:
# Plot cross sections

pot = lambda X: jax.jit(jax.vmap(potential))(X)

from src.helper import plot_cross_section
posterior     = jax.jit(lambda X: jnp.exp(-1 * pot(X)))
neg_potential = jax.jit(lambda X: -1 * pot(X))
for i in range(DoF):
    for j in range(i+1, DoF):
        print('Getting cross section for %i, %i' % (i,j))
        plot_cross_section(i, j, neg_potential, 200, true_params, a, b, param_labels)


In [ ]:
jax.vmap(potential)(prior_samples)

In [ ]:
potential(prior_samples[1])

In [ ]:
use_this(prior_samples)

In [ ]:
likelihood.evaluate(likelihood.ref_params, {})

In [ ]:
likelihood.ref_params